# nb167 v2 — Boltz2 on all 513 test compounds (P100, robust)

**v2 fix:** Do all pandas/data work BEFORE the torch downgrade (which can break numpy/pandas).
After install, only use subprocess + csv module (no pandas).

In [ ]:
# Cell 1: All data preparation FIRST, before any pip changes
import os, sys, time, subprocess, json, urllib.request, csv
from pathlib import Path

def W(msg):
    with open('/kaggle/working/trace.log', 'a') as f:
        f.write(f'[{time.strftime("%H:%M:%S")}] {msg}\n')
    print(msg, flush=True)

W('=== nb167 v2 START ===')

# Detect GPU first
import torch
cc = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
W(f'system torch={torch.__version__} cc={cc} device={torch.cuda.get_device_name(0)}')
need_downgrade = cc[0] < 7

# Download test SMILES
HF = 'https://huggingface.co/datasets/openadmet/pxr-challenge-train-test/resolve/main'
test_csv = '/kaggle/working/test.csv'
if not Path(test_csv).exists():
    urllib.request.urlretrieve(f'{HF}/pxr-challenge_TEST_BLINDED.csv', test_csv)

# Read via csv module (no pandas dep)
compounds = []
with open(test_csv) as f:
    reader = csv.DictReader(f)
    for row in reader:
        compounds.append({'name': row['Molecule Name'], 'smiles': row['SMILES']})
W(f'Loaded {len(compounds)} test compounds')

In [ ]:
# Cell 2: Write all YAMLs upfront
PXR_SEQ = 'LDRRTVVPATQHVTGTAYIWYRSGLCEHHIVEAATRGNVMTPSCKLITEELLGRPVHIVQPVKAVCSIVKQSDCRPFNQRSFKKYFTMENKVMVLNQELIKLALNFKLQDGRPHGGIIYDLSGEEDPKSWIWEVLEAWDIKAQVGPVTYAVTSLPFLQLSQYLDQDLALYIHQAFRYGPNALLDLLTDTRKHADRLELNGLAIRLLPELEVALMLLTQHTLREEKAGNFETIAEPFNALVMQVMEGYREKDPEAKQNQELHIWANKTKDPLLLEAHALDQFSCK'
YDIR = Path('/kaggle/working/yamls')
YDIR.mkdir(exist_ok=True)
ODIR = Path('/kaggle/working/outs')
ODIR.mkdir(exist_ok=True)

def safe(n): return ''.join(c if c.isalnum() else '_' for c in str(n))

for c in compounds:
    s = safe(c['name'])
    yfile = YDIR / f'{s}.yaml'
    if not yfile.exists():
        yfile.write_text(
            f'version: 1\n'
            f'sequences:\n'
            f'- protein:\n    id: A\n    sequence: {PXR_SEQ}\n'
            f'- ligand:\n    id: B\n    smiles: {c["smiles"]}\n'
            f'properties:\n- affinity:\n    binder: B\n'
        )
W(f'Wrote {len(compounds)} YAMLs')

In [ ]:
# Cell 3: NOW do the torch downgrade + boltz install (after data prep, since this can break pandas/numpy)
W('=== INSTALL ===')
if need_downgrade:
    W('Downgrading torch to 2.4.0+cu121 for P100...')
    t0 = time.time()
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall',
                        'torch==2.4.0', 'torchvision==0.19.0',
                        '--index-url', 'https://download.pytorch.org/whl/cu121'],
                       capture_output=True, text=True, timeout=1200)
    W(f'  torch downgrade rc={r.returncode} elapsed={time.time()-t0:.0f}s')
    if r.returncode != 0:
        W(f'  stderr: {r.stderr[-1500:]}')

t0 = time.time()
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'boltz'],
                   capture_output=True, text=True, timeout=1200)
W(f'  boltz install rc={r.returncode} elapsed={time.time()-t0:.0f}s')
if r.returncode != 0:
    W(f'  stderr: {r.stderr[-1500:]}')
W('=== INSTALL OK ===')

In [ ]:
# Cell 4: Main predict loop. Pure stdlib + subprocess.
env_clean = {**os.environ, 'PYTHONNOUSERSITE': '1'}

def find_aff(d):
    for jf in Path(d).rglob('*affinity*.json'):
        try:
            j = json.load(open(jf))
            return {
                'affinity_pred_value': j.get('affinity_pred_value'),
                'affinity_probability_binary': j.get('affinity_probability_binary'),
            }
        except Exception: pass
    return {}

results = []
results_csv = '/kaggle/working/nb167_test_partial.csv'
def flush_csv():
    if not results: return
    with open(results_csv, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=['name','smiles','affinity_pred_value','affinity_probability_binary'])
        w.writeheader()
        for r in results: w.writerow(r)

t_start = time.time()
for i, c in enumerate(compounds):
    name = c['name']
    s = safe(name)
    out_p = ODIR / s
    aff = find_aff(out_p) if out_p.exists() else {}
    if not aff.get('affinity_pred_value'):
        cmd = ['boltz', 'predict', str(YDIR / f'{s}.yaml'),
               '--out_dir', str(out_p), '--use_msa_server',
               '--diffusion_samples', '1', '--recycling_steps', '1', '--sampling_steps', '50']
        try:
            subprocess.run(cmd, env=env_clean, capture_output=True, text=True, timeout=900)
            aff = find_aff(out_p)
        except subprocess.TimeoutExpired:
            aff = {}
        except Exception as e:
            aff = {'_err': str(e)}
    results.append({
        'name': name,
        'smiles': c['smiles'],
        'affinity_pred_value': aff.get('affinity_pred_value'),
        'affinity_probability_binary': aff.get('affinity_probability_binary'),
    })
    if (i + 1) % 5 == 0:
        elapsed = (time.time() - t_start) / 60
        eta_h = elapsed / (i + 1) * (len(compounds) - i - 1) / 60
        n_ok = sum(1 for r in results if r.get('affinity_pred_value') is not None)
        W(f'  {i+1}/{len(compounds)} elapsed={elapsed:.0f}min ETA={eta_h:.1f}h non_nan={n_ok}/{len(results)}')
        flush_csv()

flush_csv()
n_ok = sum(1 for r in results if r.get('affinity_pred_value') is not None)
W(f'=== DONE: {n_ok}/{len(results)} compounds with affinity ===')
if n_ok > 0:
    vals = [r['affinity_pred_value'] for r in results if r.get('affinity_pred_value') is not None]
    W(f'  mean={sum(vals)/len(vals):.3f} min={min(vals):.3f} max={max(vals):.3f}')